# Bài 06: Kỹ thuật Lấy mẫu lại (Resampling Techniques)

**Mục tiêu bài học:**
- Hiểu được ý tưởng cốt lõi đằng sau việc xử lý dữ liệu mất cân bằng bằng cách thay đổi phân phối của dữ liệu huấn luyện.
- Học và áp dụng các kỹ thuật Undersampling (giảm mẫu lớp đa số) và Oversampling (tăng mẫu lớp thiểu số).
- Nắm vững các thuật toán phổ biến: Random Undersampling, Random Oversampling, và SMOTE (Synthetic Minority Over-sampling Technique).
- Hiểu được ưu, nhược điểm và các cạm bẫy khi sử dụng resampling.

## 1. Tại sao phải Resample?

Ở các bài trước, chúng ta đã thấy rằng gốc rễ của vấn đề là mô hình bị "áp đảo" bởi số lượng mẫu khổng lồ của lớp đa số và không có đủ "động lực" để học các đặc điểm của lớp thiểu số. Các chỉ số đánh giá tốt hơn giúp chúng ta đo lường vấn đề, nhưng không giải quyết nó.

**Ý tưởng của Resampling rất trực quan:** Nếu dữ liệu huấn luyện bị mất cân bằng, tại sao chúng ta không "sửa" nó trước khi đưa cho mô hình học? Chúng ta có thể làm cho tập huấn luyện trở nên cân bằng hơn bằng hai cách chính:

1.  **Undersampling (Lấy mẫu dưới):** Giảm số lượng mẫu của lớp đa số cho đến khi nó tương đương với lớp thiểu số.
2.  **Oversampling (Lấy mẫu trên):** Tăng số lượng mẫu của lớp thiểu số cho đến khi nó tương đương với lớp đa số.

![Resampling](https://i.imgur.com/g2C8VdD.png)
*Hình ảnh minh họa Undersampling và Oversampling. Nguồn: imbalanced-learn documentation*

**LƯU Ý CỰC KỲ QUAN TRỌNG:**
Resampling **CHỈ** được áp dụng trên **TẬP HUẤN LUYỆN (`X_train`, `y_train`)**. 
**KHÔNG BAO GIỜ** được áp dụng trên **TẬP KIỂM TRA (`X_test`, `y_test`)**.

Lý do là vì tập kiểm tra phải phản ánh đúng phân phối dữ liệu trong thực tế. Nếu bạn thay đổi tập kiểm tra, kết quả đánh giá của bạn sẽ trở nên vô nghĩa và không đáng tin cậy. Chúng ta muốn xem mô hình sau khi đã học từ dữ liệu "cân bằng" sẽ hoạt động tốt như thế nào trên dữ liệu "mất cân bằng" thực tế.

## 2. Chuẩn bị

Chúng ta sẽ sử dụng thư viện `imbalanced-learn` (thường được import với tên `imblearn`), một thư viện bổ trợ tuyệt vời cho Scikit-learn chuyên về xử lý dữ liệu mất cân bằng.

Nếu bạn chưa cài đặt, hãy chạy lệnh sau trong terminal hoặc notebook cell:
`!pip install imbalanced-learn`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, balanced_accuracy_score, roc_auc_score

# Import các công cụ từ imblearn
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Tạo lại bộ dữ liệu mất cân bằng
X, y = make_classification(n_samples=10000,
                           n_features=10,
                           n_informative=5,
                           n_redundant=0,
                           n_classes=2,
                           weights=[0.98, 0.02], # Tăng độ mất cân bằng lên 98/2
                           flip_y=0.05,
                           random_state=42)

# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Phân phối lớp ban đầu: {Counter(y)}")
print(f"Phân phối lớp trong tập huấn luyện: {Counter(y_train)}")
print(f"Phân phối lớp trong tập kiểm tra: {Counter(y_test)}")

## 3. Kỹ thuật Undersampling

### 3.1. Random Undersampling

**Ý tưởng:** Đây là kỹ thuật đơn giản nhất. Nó loại bỏ ngẫu nhiên các mẫu từ lớp đa số cho đến khi số lượng mẫu của nó bằng với số lượng mẫu của lớp thiểu số.

**Ưu điểm:**
- Nhanh và dễ thực hiện.
- Giảm kích thước tập huấn luyện, có thể tăng tốc độ training.

**Nhược điểm:**
- **Mất mát thông tin:** Bằng cách loại bỏ các mẫu, chúng ta có thể vô tình vứt đi những thông tin quan trọng hoặc các mẫu đại diện cho một phân nhóm nào đó trong lớp đa số. Điều này có thể làm cho ranh giới quyết định của mô hình trở nên kém tối ưu.

In [ ]:
# 1. Khởi tạo RandomUnderSampler
rus = RandomUnderSampler(random_state=42)

# 2. Áp dụng lên tập huấn luyện
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print(f"Phân phối lớp trong tập huấn luyện GỐC: {Counter(y_train)}")
print(f"Phân phối lớp sau khi Random Undersampling: {Counter(y_train_rus)}")
print(f"Kích thước tập huấn luyện GỐC: {X_train.shape}")
print(f"Kích thước tập huấn luyện sau RUS: {X_train_rus.shape}")

In [ ]:
# Huấn luyện mô hình trên dữ liệu đã undersample
model_rus = LogisticRegression(solver='liblinear', random_state=42)
model_rus.fit(X_train_rus, y_train_rus)

# Đánh giá trên tập test GỐC
y_pred_rus = model_rus.predict(X_test)

print("--- Kết quả của mô hình với Random Undersampling ---")
print(classification_report(y_test, y_pred_rus))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_rus):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_rus):.4f}")

**Nhận xét:**
So với mô hình gốc (không resampling), `Recall` của lớp 1 đã **tăng vọt** (ví dụ từ ~0.2 lên ~0.8), nhưng `Recall` của lớp 0 lại **giảm mạnh** (ví dụ từ ~0.99 xuống ~0.8). `Precision` của lớp 1 cũng có thể giảm.

Đây là một sự đánh đổi điển hình. Bằng cách loại bỏ nhiều mẫu lớp 0, mô hình trở nên ít "thiên vị" lớp 0 hơn và bắt đầu chú ý hơn đến lớp 1. Tuy nhiên, nó cũng trở nên kém chính xác hơn khi phân loại lớp 0. `Balanced Accuracy` và `ROC AUC` thường sẽ tăng lên, cho thấy đây là một sự đánh đổi có lợi.

## 4. Kỹ thuật Oversampling

### 4.1. Random Oversampling

**Ý tưởng:** Ngược lại với undersampling, kỹ thuật này tăng số lượng mẫu của lớp thiểu số bằng cách chọn ngẫu nhiên các mẫu từ lớp thiểu số và lặp lại chúng (có thay thế).

**Ưu điểm:**
- **Không mất mát thông tin:** Tất cả các mẫu trong tập huấn luyện gốc đều được giữ lại.

**Nhược điểm:**
- **Overfitting (Học vẹt):** Vì chúng ta chỉ đơn giản là sao chép các mẫu hiện có, mô hình có thể trở nên quá chuyên biệt vào chính những mẫu này và không khái quát hóa tốt trên dữ liệu mới. Nó có thể học thuộc lòng các mẫu được lặp lại thay vì học các đặc điểm chung.

In [ ]:
# 1. Khởi tạo RandomOverSampler
ros = RandomOverSampler(random_state=42)

# 2. Áp dụng lên tập huấn luyện
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

print(f"Phân phối lớp trong tập huấn luyện GỐC: {Counter(y_train)}")
print(f"Phân phối lớp sau khi Random Oversampling: {Counter(y_train_ros)}")
print(f"Kích thước tập huấn luyện GỐC: {X_train.shape}")
print(f"Kích thước tập huấn luyện sau ROS: {X_train_ros.shape}")

In [ ]:
# Huấn luyện mô hình trên dữ liệu đã oversample
model_ros = LogisticRegression(solver='liblinear', random_state=42)
model_ros.fit(X_train_ros, y_train_ros)

# Đánh giá trên tập test GỐC
y_pred_ros = model_ros.predict(X_test)

print("--- Kết quả của mô hình với Random Oversampling ---")
print(classification_report(y_test, y_pred_ros))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_ros):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_ros):.4f}")

**Nhận xét:**
Kết quả thường tương tự như Random Undersampling: `Recall` của lớp 1 tăng mạnh, `Recall` của lớp 0 giảm, và các chỉ số tổng hợp như `Balanced Accuracy` và `ROC AUC` được cải thiện. Tuy nhiên, nguy cơ overfitting là một vấn đề cần lưu ý.

### 4.2. SMOTE (Synthetic Minority Over-sampling Technique)

**Ý tưởng:** SMOTE là một cải tiến thông minh hơn của Random Oversampling. Thay vì chỉ sao chép các mẫu hiện có, SMOTE tạo ra các mẫu **tổng hợp** (synthetic) mới cho lớp thiểu số. 

**Cách hoạt động (đơn giản hóa):**
1.  Chọn một mẫu ngẫu nhiên từ lớp thiểu số (gọi là `A`).
2.  Tìm `k` hàng xóm gần nhất của `A` (cũng thuộc lớp thiểu số).
3.  Chọn ngẫu nhiên một trong số `k` hàng xóm đó (gọi là `B`).
4.  Tạo một mẫu mới bằng cách lấy một điểm ngẫu nhiên trên đường thẳng nối giữa `A` và `B`.

Bằng cách này, SMOTE tạo ra các mẫu mới nằm gần các mẫu hiện có, giúp mở rộng vùng quyết định của lớp thiểu số một cách tự nhiên hơn và giảm nguy cơ overfitting.

**Ưu điểm:**
- Giảm overfitting so với Random Oversampling.
- Tạo ra một tập dữ liệu đa dạng hơn.

**Nhược điểm:**
- Có thể tạo ra nhiễu nếu các mẫu thiểu số nằm gần các mẫu đa số, dẫn đến việc tạo ra các mẫu tổng hợp trong vùng của lớp đa số, gây ra sự chồng chéo.
- Không hiệu quả với dữ liệu có chiều cao (high-dimensional data).

In [ ]:
# 1. Khởi tạo SMOTE
smote = SMOTE(random_state=42)

# 2. Áp dụng lên tập huấn luyện
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Phân phối lớp trong tập huấn luyện GỐC: {Counter(y_train)}")
print(f"Phân phối lớp sau khi SMOTE: {Counter(y_train_smote)}")
print(f"Kích thước tập huấn luyện GỐC: {X_train.shape}")
print(f"Kích thước tập huấn luyện sau SMOTE: {X_train_smote.shape}")

In [ ]:
# Huấn luyện mô hình trên dữ liệu đã SMOTE
model_smote = LogisticRegression(solver='liblinear', random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

# Đánh giá trên tập test GỐC
y_pred_smote = model_smote.predict(X_test)

print("--- Kết quả của mô hình với SMOTE ---")
print(classification_report(y_test, y_pred_smote))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_smote):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_smote):.4f}")

**Nhận xét:**
SMOTE thường mang lại kết quả tốt nhất trong ba kỹ thuật resampling cơ bản. Nó thường đạt được sự cân bằng tốt giữa việc cải thiện `Recall` của lớp 1 mà không làm giảm quá nhiều `Recall` của lớp 0. Các chỉ số tổng hợp như `Balanced Accuracy` và `ROC AUC` thường cao nhất khi sử dụng SMOTE.

## 5. Tổng kết và Lời khuyên

| Kỹ thuật             | Ưu điểm                                       | Nhược điểm                                                              | Khi nào nên thử?                                                                                             |
|----------------------|-----------------------------------------------|-------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------|
| **Random Undersampling** | Nhanh, giảm kích thước dữ liệu.               | **Mất mát thông tin quan trọng.**                                        | Khi tập dữ liệu rất lớn và lớp đa số chiếm ưu thế áp đảo, việc giảm bớt có thể giúp tăng tốc độ huấn luyện.      |
| **Random Oversampling**  | Không mất thông tin.                          | **Nguy cơ overfitting cao.**                                             | Nhanh để thử nghiệm, nhưng thường không phải là lựa chọn tốt nhất.                                            |
| **SMOTE**              | Giảm overfitting, tạo dữ liệu đa dạng hơn.     | Có thể tạo nhiễu, không hiệu quả với dữ liệu chiều cao.                  | **Điểm khởi đầu tốt nhất** cho hầu hết các bài toán. Thường mang lại hiệu quả cao.                             |

**Pipeline trong thực tế:**
Trong một dự án thực tế, bạn nên:
1.  Xây dựng một mô hình cơ sở (baseline) trên dữ liệu gốc để có điểm so sánh.
2.  Thử nghiệm với **SMOTE** trước tiên, vì nó thường cho kết quả tốt.
3.  Nếu tập dữ liệu quá lớn, có thể thử **Random Undersampling**.
4.  Có thể kết hợp cả hai: ví dụ, sử dụng SMOTE để tăng lớp thiểu số lên một mức độ nào đó, sau đó sử dụng Undersampling để giảm nhẹ lớp đa số. `imblearn` cung cấp `SMOTEENN` và `SMOTETomek` cho mục đích này.
5.  Luôn luôn so sánh kết quả trên tập kiểm tra **gốc** để đưa ra quyết định cuối cùng.

## 6. Kết luận

Resampling là một trong những kỹ thuật mạnh mẽ và phổ biến nhất để giải quyết vấn đề mất cân bằng dữ liệu. Bằng cách thay đổi tập huấn luyện, chúng ta có thể "hướng dẫn" mô hình học tốt hơn về lớp thiểu số.

Tuy nhiên, resampling không phải là giải pháp duy nhất. Một hướng tiếp cận khác là không thay đổi dữ liệu, mà thay đổi chính thuật toán học máy để nó "nhạy cảm" hơn với chi phí của việc dự đoán sai. Đây được gọi là **Cost-Sensitive Learning**, và chúng ta sẽ tìm hiểu về nó trong bài học tiếp theo.